# 01 Data Sync and Check

**Purpose**: This notebook prepares the remote Google Drive data storage for the AIGC Research Comprehension System. Google Drive is the primary persistent data store for the project corpus.

**Rules**:
- No model training.
- No LLM calls.
- No embeddings/fine-tuning.
- Persistent data lives in Google Drive.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Define Paths

In [ ]:
from pathlib import Path
import os
import shutil
import json
import time

BASE = Path("/content/drive/MyDrive/AIGC")
NOTEBOOK_DIR = BASE / "Notebook"
DATA_DIR = BASE / "Data"

PDF_DIR = DATA_DIR / "pdfs"
TEI_DIR = DATA_DIR / "tei_xml"
DOWNLOAD_LOG_DIR = DATA_DIR / "download_logs"
PARSED_DIR = DATA_DIR / "parsed"
SECTIONS_DIR = DATA_DIR / "sections"
TABLES_DIR = DATA_DIR / "tables"
PARSE_LOG_DIR = DATA_DIR / "parse_logs"
PROBES_DIR = DATA_DIR / "probes"
REGISTRY_DIR = DATA_DIR / "registry"
REPORTS_DIR = DATA_DIR / "reports"

REPO_URL = "https://github.com/IanJ332/AIGC_Fake_Detection"
REPO_DIR = Path("/content/AIGC_Fake_Detection")

## 3. Create Folders

In [ ]:
dirs = [
    NOTEBOOK_DIR, DATA_DIR, PDF_DIR, TEI_DIR, DOWNLOAD_LOG_DIR,
    PARSED_DIR, SECTIONS_DIR, TABLES_DIR, PARSE_LOG_DIR, 
    PROBES_DIR, REGISTRY_DIR, REPORTS_DIR
]

for d in dirs:
    d.mkdir(parents=True, exist_ok=True)
    print(f"Verified: {d}")

## 4. Detect Resources

In [ ]:
import psutil
import sys
import subprocess

print(f"Python Version: {sys.version}")
print(f"RAM: {psutil.virtual_memory().total / (1024**3):.2f} GB")

try:
    gpu_name = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode().strip()
    print(f"GPU: {gpu_name}")
    print("WARNING: GPU enabled. Notebook 01 should use CPU only to save credits.")
except:
    print("GPU: None (Correct for this notebook)")

## 5. Clone Repository

In [ ]:
!rm -rf /content/AIGC_Fake_Detection
!git clone {REPO_URL} {REPO_DIR}

## 6. Install Dependencies

In [ ]:
!pip install -q pandas tqdm pyyaml pymupdf requests

## 7. Sync Metadata to Drive

In [ ]:
def sync_metadata():
    # Map: Repo File -> Drive File
    mapping = {
        REPO_DIR / "corpus/manifest_100.csv": REGISTRY_DIR / "manifest_100.csv",
        REPO_DIR / "corpus/document_registry.csv": REGISTRY_DIR / "document_registry.csv",
        REPO_DIR / "corpus/parse_registry.csv": REGISTRY_DIR / "parse_registry.csv",
        REPO_DIR / "docs/day1_status.md": REPORTS_DIR / "day1_status.md",
        REPO_DIR / "docs/day2_status.md": REPORTS_DIR / "day2_status.md",
        REPO_DIR / "docs/day3_parse_report.md": REPORTS_DIR / "day3_parse_report.md"
    }
    
    for src, dst in mapping.items():
        if src.exists():
            shutil.copy(src, dst)
            print(f"Synced: {src.name} -> Drive")
        else:
            print(f"Skipped: {src.name} (Not in repo)")

sync_metadata()

## 8. Path Configuration (Symlinks)

In [ ]:
def setup_symlinks():
    # Remove existing repo data folders if they aren't symlinks
    data_subsets = [
        ("pdfs", PDF_DIR),
        ("tei_xml", TEI_DIR),
        ("download_logs", DOWNLOAD_LOG_DIR),
        ("parsed", PARSED_DIR),
        ("sections", SECTIONS_DIR),
        ("tables", TABLES_DIR),
        ("parse_logs", PARSE_LOG_DIR)
    ]
    
    for name, drive_path in data_subsets:
        local_path = REPO_DIR / "corpus" / name
        if local_path.exists() and not local_path.is_symlink():
            if local_path.is_dir():
                shutil.rmtree(local_path)
            else:
                local_path.unlink()
        
        if not local_path.exists():
            os.symlink(drive_path, local_path)
            print(f"Linked: corpus/{name} -> Drive")

setup_symlinks()

## 9. Optional Execution

In [ ]:
RUN_DOWNLOAD = False
RUN_PARSE = False

if RUN_DOWNLOAD:
    %cd {REPO_DIR}
    !python -m src.acquire.download_documents --manifest corpus/manifest_100.csv
    !python -m src.acquire.validate_documents --registry corpus/document_registry.csv

if RUN_PARSE:
    %cd {REPO_DIR}
    !python -m src.parse.parse_pdfs --registry corpus/document_registry.csv
    !python -m src.parse.segment_sections --parsed-dir corpus/parsed
    !python -m src.parse.extract_table_candidates --sections corpus/sections/sections.jsonl
    !python -m src.parse.validate_parse

## 10. Final Metadata Sync (Back to Drive)

In [ ]:
sync_metadata() # Refresh metadata if scripts were run

## 11. Verify Drive Readiness

In [ ]:
pdf_count = len(list(PDF_DIR.glob("*.pdf")))
parsed_json_count = len(list(PARSED_DIR.glob("*.json")))
sections_exists = (SECTIONS_DIR / "sections.jsonl").exists()
tables_exists = (TABLES_DIR / "table_candidates.jsonl").exists()
manifest_exists = (REGISTRY_DIR / "manifest_100.csv").exists()

print(f"PDFs in Drive: {pdf_count}")
print(f"Parsed JSONs in Drive: {parsed_json_count}")
print(f"Sections Exists: {sections_exists}")
print(f"Tables Exists: {tables_exists}")
print(f"Manifest Exists: {manifest_exists}")

## 12. Write Status JSON

In [ ]:
commit_hash = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"]).decode().strip()

status_str = "BLOCKED"
if manifest_exists:
    status_str = "READY_FOR_DOWNLOAD"
    if pdf_count > 0:
        status_str = "READY_FOR_PARSE"
        if parsed_json_count > 0 or sections_exists:
            status_str = "READY_FOR_NOTEBOOK_02"

status_data = {
    "timestamp": time.time(),
    "repo_commit": commit_hash,
    "pdf_count": pdf_count,
    "parsed_json_count": parsed_json_count,
    "sections_exists": sections_exists,
    "tables_exists": tables_exists,
    "final_status": status_str
}

status_file = PROBES_DIR / "drive_data_status.json"
with open(status_file, "w") as f:
    json.dump(status_data, f, indent=4)

print(f"Final Status: {status_str}")
print(f"Status file saved: {status_file}")